## E3SM Monthly Output Analysis
Loc completed an E3SM model run over the Western US using NLDAS-2 forcing data. This notebook analyzes the monthly SWE output, comparing it to SNOTEL observations at a selected site.

In [8]:
import pandas as pd
import re
from buildforcing.datasets import PNNLSnotel

In [2]:
# Load SNOTEL_PATH and E3SM_OUTPUT from environment variables
import os
import sys
SNOTEL_PATH = os.getenv('SNOTEL_PATH')
E3SM_OUTPUT_PATH = os.getenv('E3SM_OUTPUT_PATH')
if not SNOTEL_PATH or not E3SM_OUTPUT_PATH:
    print("SNOTEL_PATH or E3SM_OUTPUT_PATH environment variable is not set.")
    sys.exit(1)

In [3]:
# Read in model output
model_snow = pd.read_csv(E3SM_OUTPUT_PATH + 'SWE_all_locations.csv', parse_dates=['Date'], index_col='Date')

In [5]:
# Load SNOTEL site names for fallback matching
snotel_summary = pd.read_csv(SNOTEL_PATH + 'bcqc_data_v2/SNOTEL_summary.csv')
print(f"Loaded {len(snotel_summary)} SNOTEL sites")

Loaded 829 SNOTEL sites


In [19]:
# Function to calculate RMSE from monthly output
def calculate_rmse(site):
    """
    Calculate RMSE between E3SM model output and SNOTEL observations for a given site.
    
    Parameters:
    -----------
    site : str
        Site identifier matching column name in model_snow DataFrame
        
    Returns:
    --------
    dict : Dictionary containing site name, RMSE value, bias, and number of valid observations
    """
    
    def normalize_name(name):
        """Replace all non-alphanumeric characters with underscores for matching."""
        return re.sub(r'[^a-zA-Z0-9]', '_', str(name))
    
    def find_matching_snotel_name(e3sm_name):
        """Find the actual SNOTEL name from the E3SM column name."""
        for snotel_name in snotel_summary['SNOTEL Name']:
            if normalize_name(snotel_name) == e3sm_name:
                return snotel_name
        return None
    
    try:
        # First attempt: simple underscore to space conversion
        snotel_site_name = site.replace('_', ' ')
        
        try:
            snotel = PNNLSnotel(snotel_site_name, storage_path=SNOTEL_PATH)
            snotel.load_data()
        except Exception as e:
            # Fallback: search for actual name with special characters
            matched_name = find_matching_snotel_name(site)
            if matched_name:
                snotel_site_name = matched_name
                snotel = PNNLSnotel(snotel_site_name, storage_path=SNOTEL_PATH)
                snotel.load_data()
            else:
                raise Exception(f"Could not find matching SNOTEL site for '{site}'")
        
        # Get site metadata
        latitude = snotel.latitude
        longitude = snotel.longitude
        elevation = snotel.elevation
        
        # Resample SNOTEL data to monthly, tracking missing values
        # Only calculate mean for months with 10 or fewer missing values
        swe_series = snotel.data['swe_mm']
        swe_series.index = swe_series.index.tz_localize(None)
        
        # Group by month and calculate missing count and mean
        monthly_groups = swe_series.resample('MS')
        missing_counts = monthly_groups.apply(lambda x: x.isna().sum())
        monthly_means = monthly_groups.mean()
        
        # Set months with >10 missing values to NaN
        snotel_monthly = monthly_means.copy()
        snotel_monthly.loc[missing_counts > 10] = pd.NA
        
        # Get model data for this site
        model_data = model_snow[site].copy()
        
        # Normalize E3SM dates to first of month for alignment
        model_data.index = pd.DatetimeIndex(model_data.index).to_period('M').to_timestamp()
        
        # Align the two datasets (inner join on dates)
        combined = pd.DataFrame({
            'model': model_data,
            'observed': snotel_monthly
        }).dropna()
        
        # Calculate RMSE and bias
        if len(combined) > 0:
            rmse = ((combined['model'] - combined['observed']) ** 2).mean() ** 0.5
            bias = (combined['model'] - combined['observed']).mean()
            mean_swe = combined['observed'].mean()
            n_obs = len(combined)
            n_years = n_obs / 12
            return {
                'site': site, 
                'snotel_name': snotel_site_name, 
                'latitude': latitude,
                'longitude': longitude,
                'elevation': elevation,
                'rmse': rmse, 
                'bias': bias, 
                'mean_swe': mean_swe, 
                'n_observations': n_obs, 
                'n_years': n_years
            }
        else:
            return {
                'site': site, 
                'snotel_name': snotel_site_name, 
                'latitude': latitude,
                'longitude': longitude,
                'elevation': elevation,
                'rmse': None, 
                'bias': None, 
                'mean_swe': None, 
                'n_observations': 0, 
                'n_years': 0
            }
            
    except Exception as e:
        print(f"\nError processing site {site}: {str(e)}")
        return {
            'site': site, 
            'snotel_name': None, 
            'latitude': None,
            'longitude': None,
            'elevation': None,
            'rmse': None, 
            'bias': None, 
            'mean_swe': None, 
            'n_observations': 0, 
            'n_years': 0, 
            'error': str(e)
        }

In [16]:
# Testing
quemazon = PNNLSnotel('Quemazon', storage_path=SNOTEL_PATH)
quemazon.load_data()
quemazon.data.head()

,T_max_C,T_min_C,T_avg_C,precip_mm,swe_mm
Date,,,,,
1980-06-04 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-05 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-06 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-07 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN
1980-06-08 00:00:00-06:00,NaN,NaN,NaN,0.0,NaN


In [18]:
quemazon.latitude

np.float64(35.9219513)

In [ ]:
# Test the function on Quemazon
result = calculate_rmse('Quemazon')
print(result)

In [20]:
result = calculate_rmse('Aneroid_Lake__2')
print(result)

{'site': 'Aneroid_Lake__2', 'snotel_name': 'Aneroid Lake #2', 'latitude': 45.21, 'longitude': -117.19, 'elevation': 2256.0975609756097, 'rmse': np.float64(103.9714628790538), 'bias': np.float64(25.164294061522103), 'mean_swe': np.float64(215.81815344302544), 'n_observations': 228, 'n_years': 19.0}


In [21]:
# Get all site names from model output columns
sites = model_snow.columns.tolist()
print(f"Found {len(sites)} sites in model output")

Found 782 sites in model output


In [22]:
# Calculate RMSE for each site
results = []
for i, site in enumerate(sites, 1):
    print(f"\rProcessing {i}/{len(sites)}: {site[:40]:<40}", end='', flush=True)
    result = calculate_rmse(site)
    results.append(result)

print("\n")  # Move to new line when done

# Create results dataframe
results_df = pd.DataFrame(results)
print(f"Processed {len(results_df)} sites")
results_df

Processing 782/782: Upper_San_Juan                          

Processed 782 sites


,site,snotel_name,latitude,longitude,elevation,rmse,bias,mean_swe,n_observations,n_years
0,Forestdale_Creek,Forestdale Creek,38.682311,-119.960653,2445.060059,212.404308,-125.741251,283.930242,183,15.250000
1,Horse_Meadow,Horse Meadow,38.836411,-119.888317,2623.053223,82.964004,-42.224436,122.707259,123,10.250000
2,Burnside_Lake,Burnside Lake,38.719328,-119.895194,2491.674316,63.934598,-31.358075,199.963069,195,16.250000
3,Summit_Meadow,Summit Meadow,38.397400,-119.536217,2843.777344,55.677242,-31.787642,100.512534,96,8.000000
4,Carson_Pass,Carson Pass,38.692667,-120.002128,2547.690674,136.200060,-74.725528,251.335935,159,13.250000
...,...,...,...,...,...,...,...,...,...,...
777,Cole_Canyon,Cole Canyon,44.490000,-104.410000,1801.829268,52.040724,-29.363180,37.169551,228,19.000000
778,Taylor_Green,Taylor Green,45.080000,-117.550000,1750.000000,160.681404,-93.949861,137.235443,223,18.583333
779,Hand_Creek,Hand Creek,48.310000,-114.840000,1535.060976,57.305851,-30.079824,76.280573,228,19.000000
780,Santiam_Jct_,Santiam Jct.,44.440000,-121.950000,1140.243902,112.458875,-45.143419,89.990578,228,19.000000


In [ ]:
# Display summary statistics
print("RMSE Summary Statistics:")
print(results_df['rmse'].describe())
print(f"\nSites with valid RMSE: {results_df['rmse'].notna().sum()}")
print(f"Sites with errors: {results_df['rmse'].isna().sum()}")

In [23]:
# Save results to CSV with rounded values
results_output = results_df.copy()
results_output['elevation'] = results_output['elevation'].round(1)
results_output['rmse'] = results_output['rmse'].round(1)
results_output['bias'] = results_output['bias'].round(1)
results_output['mean_swe'] = results_output['mean_swe'].round(1)

results_output.to_csv('e3sm_snotel_comparison.csv', index=False)
print("Results saved to e3sm_snotel_comparison.csv")

Results saved to e3sm_snotel_comparison.csv
